In [3]:
# %%
# STEP 2 — IMPORT LIBRARIES
# --------------------------------------------------
import pandas as pd
from pathlib import Path

In [ ]:
# %%
# STEP 3 — DEFINE BASE DIRECTORY AND GET BUILDING LIST
# --------------------------------------------------
base_dir = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

# Get all building folders (only directories)
building_dirs = [d for d in base_dir.iterdir() if d.is_dir()]

print(f"Found {len(building_dirs)} building folders.")

Found 1 building folders.


In [ ]:
# %%
# STEP 4 — DEFINE COLUMN MAPPING
# --------------------------------------------------
column_mapping = {
    # wood_logs
    "wood_logs_tot_1": "wood_logs_tot_1",
    "wood_logs_tot_2": "wood_logs_tot_2",
    "wood_logs_tot_3": "wood_logs_tot_3",
    "wood_logs_tot_4": "wood_logs_tot_4",
    "wood_logs_tot_5": "wood_logs_tot_5",
    "wood_logs_tot_6": "wood_logs_tot_6",
    "wood_logs_tot_7": "wood_logs_tot_7",
    "wood_logs_tot_8": "wood_logs_tot_8",
    "wood_logs_tot_9": "wood_logs_tot_9",
}

print("Column mapping loaded.")

Column mapping loaded.


In [6]:
# %%
# STEP 5 — LOOP THROUGH ALL BUILDINGS
# --------------------------------------------------
success_count = 0
fail_count = 0

for building_dir in building_dirs:
    
    building_id = building_dir.name
    print(f"\nProcessing building: {building_id}")
    
    heating_file = building_dir / "TOTAL" / "energy_results.csv"
    energy_file = building_dir / "TOTAL" / "energy_results_tot.csv"
    
    # Check files exist
    if not heating_file.exists() or not energy_file.exists():
        print("Missing required files. Skipping.")
        fail_count += 1
        continue
    
    try:
        # Load data
        heat_df = pd.read_csv(heating_file)
        energy_df = pd.read_csv(energy_file)
        
        # Validate columns
        missing_source = [col for col in column_mapping if col not in heat_df.columns]
        missing_target = [col for col in column_mapping.values() if col not in energy_df.columns]
        
        if missing_source or missing_target:
            print(f"Missing columns. Source: {missing_source}, Target: {missing_target}")
            fail_count += 1
            continue
        
        # Copy data safely
        for source_col, target_col in column_mapping.items():
            
            source_values = heat_df[source_col].values
            target_length = len(energy_df)
            source_length = len(source_values)
            
            # Initialize column
            energy_df[target_col] = pd.NA
            
            # Safe copy length
            copy_length = min(source_length, target_length)
            
            energy_df.loc[:copy_length-1, target_col] = source_values[:copy_length]
        
        # Save updated file
        energy_df.to_csv(energy_file, index=False)
        
        print("Success")
        success_count += 1
    
    except Exception as e:
        print(f"Error: {e}")
        fail_count += 1

print("\n--- SUMMARY ---")
print(f"Successful: {success_count}")
print(f"Failed: {fail_count}")


Processing building: 1001664924
Success

--- SUMMARY ---
Successful: 1
Failed: 0
